In [ ]:
import sys
import os

import matplotlib.pyplot as plt
import seaborn as sns
import copy
from sklearn.metrics import r2_score

# sys.path.append(os.path.abspath('..'))

from utils.feat_selection import *

In [ ]:
from pathlib import Path
MAIN_DIR = Path().resolve().parent.parent
PATH_DATA = MAIN_DIR / 'data' / 'cleaned'
# where save the data after feature selection
PATH_FEAT_SELECT_DATA = MAIN_DIR / 'data' / 'feature_selection'

RANDOM_STATE = 42
THRESHOLD_CORR_FEATURES = 0.7
MAX_FEATURES = 80
CV_FOLDS = 5
MODEL = ...

In [13]:
DESCRIPTOR_NAMES = []

# Feature Selection
Since we've an huge number of features, we need to select the most relevant ones for our task. This will help us reduce overfitting, improve model performance and decrease training time.

# Dataset load

In [ ]:
X_train = ...
y_train = ...
X_test = ...
y_test = ...

# Filter method (Spearman)

In [ ]:
# Compute Spearman correlation matrix
corr = spearman_corr_matrix(X_train)

# Plot heatmap
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=False, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
)
ax.set_title('Spearman correlation matrix — all features')
plt.tight_layout()
plt.show()

In [ ]:
corr_with_target = (
    spearmanr(X_train, y_train)[0][:-1, -1]
)
filter_ranking = pd.Series(
    np.abs(corr_with_target),
    index=DESCRIPTOR_NAMES,
).sort_values(ascending=False)

selected_filter = filter_ranking.index[:MAX_FEATURES].tolist()

print("Filter — selected features (in rank order):")
print(selected_filter)

# Forward selection (FS)

In [ ]:
selected_fs, scores_fs = forward_selection(
    model = MODEL
    X_train = X_train,
    y_train = y_train,
    max_features=MAX_FEATURES,
    cv=CV_FOLDS,
)
opt_nr_var_fs = np.argmax(scores_fs)
best_selected_fs = selected_fs[:opt_nr_var_fs]
best_scores_fs = selected_fs[:opt_nr_var_fs]

print("Forward selection — selected features (in order of selection):")
print(selected_fs)
print(f"\nNumber of optimal var selected: {opt_nr_var_fs}")

# Cluestered forward selection (CFS)

In [ ]:
# Method 3: Clustered forward selection

selected_cfs, scores_cfs, cluster_labels = clustered_forward_selection(
    model = MODEL
    X_train, y_train,
    corr_matrix=corr,
    corr_threshold=THRESHOLD_CORR_FEATURES,
    max_features=MAX_FEATURES,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
)

opt_nr_var_cfs = np.argmax(scores_cfs)
best_selected_cfs = selected_cfs[:opt_nr_var_cfs]
best_scores_cfs = scores_cfs[:opt_nr_var_cfs]

print("Clustered forward selection — selected features (in order of selection):")
print(selected_cfs)
print(f"\nNumber of clusters found: {len(set(cluster_labels.values()))}")
print(f"\nNumber of optimal var selected: {opt_nr_var_cfs}")

# Feature selection performances

In [ ]:
model_filter = copy(MODEL)
model_filter.fit(X_train[selected_filter], y_train)
r2_filter = r2_score(y_test, model_filter.predict(X_test[selected_filter]))

model_fs = copy(MODEL)
model_fs.fit(X_train[best_selected_fs], y_train)
r2_fs = r2_score(y_test, model_fs.predict(X_test[best_selected_fs]))

model_cfs = copy(MODEL)
model_cfs.fit(X_train[best_selected_cfs], y_train)
r2_cfs = r2_score(y_test, model_cfs.predict(X_test[best_selected_cfs]))

print(f"R² (filter {len(selected_filter)} features) : {r2_filter:.3f}")
print(f"R² (FS {len(best_selected_fs)} features) : {r2_fs:.3f}")
print(f"R² (CFS {len(selected_cfs)} features) : {r2_cfs:.3f}")

# Permutation feature importance (PFI)

In [ ]:
df_imp_filter = permutation_importance_df(
    model_filter, X_test[selected_filter], y_test, random_state=RANDOM_STATE
)
df_imp_fs = permutation_importance_df(
    model_fs, X_test[best_selected_fs], y_test, random_state=RANDOM_STATE
)
df_imp_cfs = permutation_importance_df(
    model_cfs, X_test[best_selected_cfs], y_test, random_state=RANDOM_STATE
)

# 3. Side-by-side plot
fig, axes = plt.subplots(1, 4, figsize=(13, 20))

for ax, df_imp, title in zip(
    axes,
    [df_imp_filter, df_imp_fs, df_imp_cfs],
    [f'PFI — {len(selected_filter)} Filter features',
     f'PFI — {len(best_selected_fs)} Forward features',
     f'PFI — {len(best_selected_cfs)} CFS features'],
):
    ax.barh(
        df_imp['feature'],
        df_imp['importance_mean'],
        xerr=df_imp['importance_std'],
        align='center', color='steelblue', ecolor='gray', capsize=3,
    )
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
    ax.set_xlabel('Permutation importance (R² drop)')
    ax.set_title(title)
    ax.set_ylim(axes[0].get_ylim())

plt.tight_layout()
plt.show()

## Export Data

In [ ]:
# Create output directory if needed
os.makedirs(PATH_FEAT_SELECT_DATA, exist_ok=True)

...